# delbina-sum: train + eval

Fine-tunes **Qwen2.5-0.5B-Instruct** with LoRA on dataset v4 (targets built from the planted facts,
correct by construction), then scores base vs tuned on the held-out test split (identical to v2's, so v2, v3 and v4 compare directly) (and tuned on the frozen v0 test split) and pushes the
model and data to Hugging Face (both **private**; you flip them public after reading the numbers).

**Runtime**: Kaggle (GPU T4 x2, Internet on, `HF_TOKEN` secret attached) or Colab → Runtime ▸ Change runtime type ▸ T4 GPU. Needs the `HF_TOKEN` secret (write access).
Run cells top to bottom. Rough times on one T4: install 2 min, training ~40-60 min, baseline eval ~10 min, tuned eval ~10 min.

In [ ]:
# --- settings ---
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_LEN = 6144          # tokens, prompt + target; longer samples are dropped (counted below)
EPOCHS = 2
LR = 2e-4
LORA_R, LORA_ALPHA = 16, 32
GRAD_ACCUM = 16         # effective batch 16 = 1 per GPU x n_gpus x (16 / n_gpus)
EVAL_BATCH = 8
MAX_NEW_TOKENS = 384    # targets are <= ~200 tokens
BAR = 15.0              # tuned fact recall must beat base by this many points
INVENTED_MAX = 2.0      # % of fact lines whose value is not in the chat
UNSEEN_MIN = 70.0       # recall on fact kinds held out from training
DISTRACTOR_MAX = 5.0    # % of chats with a rejected value that keep it
V0_UNSEEN_MIN = 60.0    # frozen v0 test (never generated by v2/v3): unseen-kind recall
V0_INVENTED_MAX = 2.0
HF_MODEL_REPO = "delbina/delbina-sum"
HF_DATA_REPO = "delbina/delbina-sum-data"
VERSION = "v4"
PUSH = True             # private uploads; nothing goes public from here

In [ ]:
# --- platform + token (Colab or Kaggle) ---
import os
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    WORK = "/kaggle/working"
else:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    WORK = "/content"
print("token loaded:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# --- install (pinned: the code was tested against these versions) ---
!pip install -q "transformers==4.57.6" "peft==0.17.1" accelerate "delbina==0.4.0" huggingface_hub
!rm -rf {WORK}/delbina-sum && git clone -q https://github.com/delbina/delbina-sum.git {WORK}/delbina-sum
!cd {WORK}/delbina-sum && git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- data: regenerate VERSION (deterministic, same bytes as data/<VERSION> on the laptop) + frozen v0 test ---
import sys, json
sys.path[:0] = [f"{WORK}/delbina-sum/datagen", f"{WORK}/delbina-sum/train"]
import importlib
md = importlib.import_module(f"make_dataset_{VERSION}")
import make_dataset as md0
import sft
from common import score

data = {s: [md.make_sample(s, i) for i in range(n)] for s, n in {"train": 2000, "val": 200, "test": 300}.items()}
test_v0 = [md0.make_sample("test", i) for i in range(300)]
print({k: len(v) for k, v in data.items()}, "v0 test", len(test_v0))
print(data["train"][0]["target"])

In [ ]:
# --- tokenizer + token lengths ---
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(BASE)
train_ex, dropped = sft.build_examples(tok, data["train"], MAX_LEN)
lens = sorted(len(e["input_ids"]) for e in train_ex)
print(f"train examples {len(train_ex)}, dropped (> {MAX_LEN} tokens) {dropped}")
print("tokens p50", lens[len(lens)//2], "p95", lens[int(len(lens)*.95)], "max", lens[-1], "total", sum(lens))

In [ ]:
# --- train LoRA (both GPUs on Kaggle T4 x2 via torchrun; one GPU on Colab) ---
# runs BEFORE any model is loaded in this notebook, so the GPUs are empty for training
import subprocess, time
NGPU = torch.cuda.device_count()
OUT = f"{WORK}/out"
script = f"{WORK}/delbina-sum/train/ddp_train.py"
args = ["--version", VERSION, "--out", OUT, "--max-len", str(MAX_LEN), "--epochs", str(EPOCHS), "--lr", str(LR),
        "--lora-r", str(LORA_R), "--lora-alpha", str(LORA_ALPHA)]
cmd = ([sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={NGPU}", script]
       if NGPU > 1 else [sys.executable, script]) + args
print("GPUs:", NGPU, "|", " ".join(cmd))
t = time.time()
subprocess.run(cmd, check=True, env={**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})   # progress prints below; check=True stops the notebook if training fails
print(f"trained in {(time.time() - t) / 60:.0f} min on {NGPU} GPU(s)")


In [ ]:
# --- baseline: plain 0.5B on the test split ---
import time
def run_eval(model, name, rows=None):
    rows = rows or data["test"]
    t = time.time()
    outs = sft.summarize(model, tok, rows, batch_size=EVAL_BATCH, max_new_tokens=MAX_NEW_TOKENS)
    res = sft.evaluate(rows, outs, md.TEST_KINDS | md0.TEST_KINDS)
    print(f"{name}: {time.time() - t:.0f}s")
    print(json.dumps(res, indent=1))
    return outs, res

base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="cuda")
base_outs, BASE_RES = run_eval(base, "base")
print("-" * 80); print(base_outs[0]["summary"][:800])
del base; torch.cuda.empty_cache()

In [ ]:
# --- tuned eval ---
from pathlib import Path
from peft import PeftModel
OUT = Path(WORK) / "out"                                     # adapter saved there by ddp_train.py (~35 MB)
base_fp32 = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32, device_map="cuda")
tuned = PeftModel.from_pretrained(base_fp32, OUT / "adapter").merge_and_unload().half().eval()
tuned.config.use_cache = True
tuned_outs, TUNED_RES = run_eval(tuned, "tuned")
tuned_v0_outs, TUNED_V0 = run_eval(tuned, "tuned on v0 test", test_v0)  # v1: unseen 43.1, invented 2.0 | v2: unseen 67.2, invented 2.7
for r, o in list(zip(data["test"], tuned_outs))[:3]:
    print("=" * 80); print("facts:", [(f["subject"], f["value"]) for f in r["facts"]]); print(o["summary"])

In [ ]:
# --- verdict ---
gain = TUNED_RES["fact_recall"] - BASE_RES["fact_recall"]
rows = ["fact_recall", "bound_recall", "unseen_recall", "invented_rate", "invented_rate_ci", "invented_strict_rate", "invented_chats",
        "stale_hit_rate", "distractor_hit_rate", "perfect_rate", "truncated", "summary_chars_p50"]
print(f"{'metric':20} {'base':>8} {'tuned':>8}")
for k in rows:
    print(f"{k:20} {BASE_RES[k]!s:>8} {TUNED_RES[k]!s:>8}")
held = {k: (BASE_RES["recall_by_kind"].get(k), TUNED_RES["recall_by_kind"].get(k)) for k in md.TEST_KINDS}
print("held-out kinds (base, tuned):", held)
checks = {f"recall gain >= +{BAR}": gain >= BAR,
          f"invented <= {INVENTED_MAX}%": TUNED_RES["invented_rate"] <= INVENTED_MAX,
          f"unseen-kind recall >= {UNSEEN_MIN}": (TUNED_RES["unseen_recall"] or 0) >= UNSEEN_MIN,
          f"distractor hits <= {DISTRACTOR_MAX}%": (TUNED_RES["distractor_hit_rate"] or 0) <= DISTRACTOR_MAX,
          f"v0 test unseen-kind recall >= {V0_UNSEEN_MIN}": (TUNED_V0["unseen_recall"] or 0) >= V0_UNSEEN_MIN,
          f"v0 test invented <= {V0_INVENTED_MAX}%": TUNED_V0["invented_rate"] <= V0_INVENTED_MAX}
print(f"\ngain {gain:+.1f} pts")
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print("v0 test (tuned):", {k: TUNED_V0[k] for k in ("fact_recall", "unseen_recall", "invented_rate", "invented_rate_ci", "invented_strict_rate", "stale_hit_rate")})
print("v0 test held-out kinds:", {k: TUNED_V0["recall_by_kind"].get(k) for k in sorted(md0.TEST_KINDS)})
print("recall by how (test):", TUNED_RES["recall_by_how"])
PASS = all(checks.values())
print("RELEASE CANDIDATE" if PASS else "HOLD")

In [ ]:
# --- save model, eval files and model card ---
tuned.save_pretrained(OUT / "model", safe_serialization=True)
tok.save_pretrained(OUT / "model")
with open(OUT / "model" / "eval_test.jsonl", "w", encoding="utf-8") as fh:
    for r, b, t_ in zip(data["test"], base_outs, tuned_outs):
        fh.write(json.dumps({"id": r["id"], "facts": r["facts"], "base": b["summary"], "tuned": t_["summary"]},
                            ensure_ascii=False) + "\n")
with open(OUT / "model" / "eval_test_v0.jsonl", "w", encoding="utf-8") as fh:  # for diagnosing v0-test misses
    for r, t_ in zip(test_v0, tuned_v0_outs):
        fh.write(json.dumps({"id": r["id"], "facts": r["facts"], "tuned": t_["summary"]}, ensure_ascii=False) + "\n")
RESULTS = {"version": VERSION, "base_model": BASE, "base": BASE_RES, "tuned": TUNED_RES, "tuned_on_v0_test": TUNED_V0,
           "gain_pts": round(gain, 1), "checks": checks,
           "train": {"examples": len(train_ex), "dropped": dropped, "epochs": EPOCHS, "lr": LR,
                     "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "effective_batch": GRAD_ACCUM, "n_gpus": NGPU, "max_len": MAX_LEN}}
(OUT / "model" / "results.json").write_text(json.dumps(RESULTS, indent=1))
card = open(f"{WORK}/delbina-sum/train/model_card.md", encoding="utf-8").read()
fill = {"BASE_RECALL": BASE_RES["fact_recall"], "TUNED_RECALL": TUNED_RES["fact_recall"],
        "BASE_BOUND": BASE_RES["bound_recall"], "TUNED_BOUND": TUNED_RES["bound_recall"],
        "BASE_STALE": BASE_RES["stale_hit_rate"], "TUNED_STALE": TUNED_RES["stale_hit_rate"],
        "BASE_PERFECT": BASE_RES["perfect_rate"], "TUNED_PERFECT": TUNED_RES["perfect_rate"],
        "BASE_CHARS": BASE_RES["summary_chars_p50"], "TUNED_CHARS": TUNED_RES["summary_chars_p50"],
        "HELD_BASE": " / ".join(str(held[k][0]) for k in sorted(held)), "HELD_TUNED": " / ".join(str(held[k][1]) for k in sorted(held)),
        "GAIN": f"{gain:+.1f}", "N_TRAIN": len(train_ex), "EPOCHS": EPOCHS, "LORA_R": LORA_R}
for k, v in fill.items():
    card = card.replace("{{" + k + "}}", str(v))
(OUT / "model" / "README.md").write_text(card, encoding="utf-8")
print(sorted(p.name for p in (OUT / "model").iterdir()))

In [ ]:
# --- push (private) ---
from huggingface_hub import HfApi
if not PUSH:
    print("PUSH=False: nothing uploaded")
else:
    api = HfApi()
    api.create_repo(HF_MODEL_REPO, private=True, exist_ok=True)
    api.create_branch(HF_MODEL_REPO, branch=VERSION, exist_ok=True)  # main keeps the last release candidate
    api.upload_folder(folder_path=str(OUT / "model"), repo_id=HF_MODEL_REPO, revision=VERSION,
                      commit_message=f"{VERSION}: LoRA merged, fact recall {TUNED_RES['fact_recall']} (base {BASE_RES['fact_recall']})")
    api.upload_folder(folder_path=str(OUT / "adapter"), path_in_repo="adapter", repo_id=HF_MODEL_REPO, revision=VERSION,
                      commit_message=f"{VERSION}: LoRA adapter")
    data_dir = OUT / "data" / VERSION
    data_dir.mkdir(parents=True, exist_ok=True)
    for s, rows in data.items():
        with open(data_dir / f"{s}.jsonl", "w", encoding="utf-8") as fh:
            for r in rows:
                fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    api.create_repo(HF_DATA_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(data_dir), path_in_repo=VERSION, repo_id=HF_DATA_REPO, repo_type="dataset",
                      commit_message=f"{VERSION}: synthetic chats + correct-by-construction targets")
    print(f"model: https://huggingface.co/{HF_MODEL_REPO}/tree/{VERSION}")
    print(f"data:  https://huggingface.co/datasets/{HF_DATA_REPO}")